In [ ]:
# === KONFIGURACJA ===
# ZMIEŃ TO NA 0, 1, 2, 3 LUB 4 PRZED KAŻDĄ SESJĄ
FOLD_IDX = 4

# Hiperparametry
N_FOLDS = 5
RANDOM_SEED = 42
NUM_CLASSES = 15
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Kompensacja małolicznych klas
USE_CLASS_WEIGHTS = True

# Trenowanie
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 20
PHASE1_LR = 1e-3
PHASE2_LR = 1e-4
PATIENCE_P1 = 3
PATIENCE_P2 = 5

# Liczba warstw odblokowanych w fazie 2
RESNET_UNFREEZE = 20
VGG_UNFREEZE = 8

# Czy zapisać wagi modeli (potrzebne do Grad-CAM)
SAVE_WEIGHTS = True

# === KAGGLE API ===
# Token z https://www.kaggle.com/settings/account -> API -> Create New Token
KAGGLE_USERNAME = 'USERNAME'
KAGGLE_KEY = 'TOKEN-KEY'  # np. 'KGAT_4b2cf...'

# Ścieżki na Drive (tylko zapis wyników)
DRIVE_BASE = '/content/drive/MyDrive/plant_disease_classification'
RESULTS_DIR = f'{DRIVE_BASE}/results'

# Lokalna ścieżka do datasetu (pobierane z Kaggle API przy każdej sesji, ~30s)
LOCAL_DATA = '/content/plantvillage'
SRC = f'{LOCAL_DATA}/PlantVillage'


In [ ]:
# === MONTOWANIE DRIVE ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(DRIVE_BASE, exist_ok=True)
FOLD_DIR = f'{RESULTS_DIR}/fold_{FOLD_IDX}'
os.makedirs(FOLD_DIR, exist_ok=True)

print(f'Drive zamontowany, wyniki będą zapisane w: {FOLD_DIR}')


In [ ]:
# == KAGGLE ==
if KAGGLE_USERNAME == 'TWOJA_NAZWA_KAGGLE' or KAGGLE_KEY == 'WKLEJ_TOKEN_TUTAJ':
    raise ValueError(
        'Wklej swoj username i token Kaggle w komorce 2 (KAGGLE_USERNAME i KAGGLE_KEY). '
        'Token: https://www.kaggle.com/settings/account -> API -> Create New Token'
    )

# Zmienne srodowiskowe
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

# Pobieranie datasetu jesli jeszcze go nie ma lokalnie
if not os.path.exists(SRC):
    print('Pobieram dataset PlantVillage z Kaggle...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    ret = os.system(f'kaggle datasets download -d emmarex/plantdisease -p {LOCAL_DATA} --unzip --quiet')
    if ret != 0:
        raise RuntimeError('Pobieranie nieudane. Sprawdz czy username i token sa poprawne.')
    print(f'Gotowe. Lista folderow: {sorted(os.listdir(SRC))[:3]}...')
else:
    print(f'Dataset juz pobrany lokalnie: {SRC}')


In [ ]:
# === IMPORTY ===
import json, time, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                              balanced_accuracy_score, f1_score,
                              precision_recall_fscore_support)
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (GlobalAveragePooling2D, Dense, Dropout,
                                      BatchNormalization)
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                         ReduceLROnPlateau)

tf.keras.utils.set_random_seed(RANDOM_SEED)

print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU: {gpus}')


In [ ]:
# === SPRAWDZENIE STANU PODZIAŁU ===
# Marker pełnego zakończenia podziału
done_marker = f'{FOLD_DIR}/fold_{FOLD_IDX}_done.json'

# Markery zakończenia poszczególnych modeli
resnet_done = os.path.exists(f'{FOLD_DIR}/resnet50_metrics.json')
vgg_done = os.path.exists(f'{FOLD_DIR}/vgg16_metrics.json')

if os.path.exists(done_marker):
    print(f'Podział {FOLD_IDX+1} jest już ukończony w całości.')
    print(f'Jeśli chcesz powtórzyć, usuń: {done_marker}')
else:
    print(f'Rozpoczynam podział {FOLD_IDX+1} z {N_FOLDS}')
    if resnet_done:
        print('  - ResNet50: gotowe (pomijam)')
    else:
        print('  - ResNet50: do trenowania')
    if vgg_done:
        print('  - VGG16: gotowe (pomijam)')
    else:
        print('  - VGG16: do trenowania')


In [ ]:
# === BUDOWA INDEKSU PLIKÓW ===
# Sortowanie list katalogów i plików

records = []
for cls in sorted(os.listdir(SRC)):
    cls_path = os.path.join(SRC, cls)
    if not os.path.isdir(cls_path):
        continue
    for f in sorted(os.listdir(cls_path)):
        records.append({'filepath': os.path.join(cls_path, f), 'class': cls})

df = pd.DataFrame(records)
print(f'Łącznie obrazów: {len(df)}')
print(f'Liczba klas: {df["class"].nunique()}')
print()
print('Liczność klas:')
print(df['class'].value_counts().sort_index())

assert df['class'].nunique() == NUM_CLASSES, \
    f'Oczekiwano {NUM_CLASSES} klas, znaleziono {df["class"].nunique()}'


In [ ]:
# === walidacja krzyżowa z 5 podziałami ===
# Każdy podział: 80% trening + 10% walidacja + 10% test

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_splits = list(skf.split(df, df['class']))

train_val_idx, test_idx = fold_splits[FOLD_IDX]

# Dalszy podział train_val na trening/walidację 88,9/11,1 to globalnie ok. 80/10
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.111,
    stratify=df.iloc[train_val_idx]['class'],
    random_state=RANDOM_SEED
)

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val = df.iloc[val_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

print(f'Podział {FOLD_IDX+1}:')
print(f'  Trening:    {len(df_train):>6} ({len(df_train)/len(df)*100:.1f}%)')
print(f'  Walidacja:  {len(df_val):>6} ({len(df_val)/len(df)*100:.1f}%)')
print(f'  Test:       {len(df_test):>6} ({len(df_test)/len(df)*100:.1f}%)')

# Zapisz indeksy podziału
np.savez(f'{FOLD_DIR}/split_indices.npz',
         train=train_idx, val=val_idx, test=test_idx)


In [ ]:
# === WAGI KLAS i kompensacja małolicznych klas ===
# Strategia 'balanced': waga klasy = n_samples / (n_classes * n_samples_class)
# Dla Potato healthy (152 obrazy): waga ok. 8-9x
# Dla Tomato YellowLeafCurl (3209): waga ok. 0,4x

class_names_sorted = sorted(df_train['class'].unique())
class_to_idx = {c: i for i, c in enumerate(class_names_sorted)}
y_train_int = df_train['class'].map(class_to_idx).values

if USE_CLASS_WEIGHTS:
    weights = compute_class_weight(
        'balanced',
        classes=np.arange(NUM_CLASSES),
        y=y_train_int,
    )
    class_weights_dict = dict(enumerate(weights))
    print('Wagi klas:')
    for cls, idx in class_to_idx.items():
        n = (y_train_int == idx).sum()
        print(f'  [{idx:2d}] {cls:<35} n={n:>5}  waga={class_weights_dict[idx]:.3f}')
else:
    class_weights_dict = None
    print('Wagi klas wyłączone.')

# Zapisuje wagi do pliku
if class_weights_dict:
    with open(f'{FOLD_DIR}/class_weights.json', 'w') as f:
        json.dump({
            'strategy': 'balanced',
            'classes': class_names_sorted,
            'weights': {class_names_sorted[i]: float(w)
                        for i, w in class_weights_dict.items()},
        }, f, indent=2)


In [ ]:
# === FUNKCJA POMOCNICZA generatory danych ===
def make_generators(preprocess_fn, batch_size=BATCH_SIZE):
    train_gen = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        zoom_range=0.2,
    ).flow_from_dataframe(
        df_train, x_col='filepath', y_col='class',
        target_size=IMG_SIZE, batch_size=batch_size,
        class_mode='categorical', shuffle=True, seed=RANDOM_SEED,
    )
    val_gen = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
    ).flow_from_dataframe(
        df_val, x_col='filepath', y_col='class',
        target_size=IMG_SIZE, batch_size=batch_size,
        class_mode='categorical', shuffle=False,
    )
    test_gen = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
    ).flow_from_dataframe(
        df_test, x_col='filepath', y_col='class',
        target_size=IMG_SIZE, batch_size=batch_size,
        class_mode='categorical', shuffle=False,
    )
    return train_gen, val_gen, test_gen


In [ ]:
# === FUNKCJA POMOCNICZA budowa modelu z dwufazowym uczeniem transferowym ===
def build_and_train(model_name, base_class, preprocess_fn, n_unfreeze):
    """Trenuj jeden model dwufazowym uczeniem transferowym.

    Zwraca: model, eval_results
    """
    print(f'\n{"="*60}\n  {model_name}: podział {FOLD_IDX+1}\n{"="*60}')
    t_start = time.time()

    train_gen, val_gen, test_gen = make_generators(preprocess_fn)

    # Architektura
    base = base_class(weights='imagenet', include_top=False,
                      input_shape=(*IMG_SIZE, 3))
    base.trainable = False

    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        BatchNormalization(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax'),
    ])

    # FAZA 1
    print(f'\n--- {model_name}: faza 1 (zamrożony model bazowy) ---')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    h1 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=PHASE1_EPOCHS, verbose=2,
        class_weight=class_weights_dict,
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=PATIENCE_P1,
                          restore_best_weights=True, verbose=1),
        ],
    )

    # FAZA 2 odblokowanie ostatnich warstw
    
    print(f'\n--- {model_name}: faza 2 (dostrojenie ostatnich {n_unfreeze} warstw) ---')
    base.trainable = False
    n_bn_skipped = 0
    n_unfrozen = 0
    for layer in base.layers[-n_unfreeze:]:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            n_bn_skipped += 1
        else:
            layer.trainable = True
            n_unfrozen += 1
    print(f'Odblokowano {n_unfrozen} warstw, pominięto {n_bn_skipped} warstw BatchNormalization')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    ckpt_path = f'{FOLD_DIR}/{model_name.lower()}_best.h5'
    h2 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=PHASE2_EPOCHS, verbose=2,
        class_weight=class_weights_dict,
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=PATIENCE_P2,
                          restore_best_weights=True, verbose=1),
            ModelCheckpoint(ckpt_path, monitor='val_accuracy',
                            save_best_only=True, verbose=1),
            ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                              patience=3, min_lr=1e-7, verbose=1),
        ],
    )

    # OCENA na zbiorze testowym
    print(f'\n--- {model_name}: ocena na zbiorze testowym ---')
    test_gen.reset()
    test_loss, test_acc = model.evaluate(test_gen, verbose=1)

    test_gen.reset()
    y_proba = model.predict(test_gen, verbose=1)
    y_pred = np.argmax(y_proba, axis=1)
    y_true = test_gen.classes
    class_names = list(test_gen.class_indices.keys())

    # Metryki
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    precision, recall, f1_per_class, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=range(NUM_CLASSES), zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))

    elapsed = (time.time() - t_start) / 60
    print(f'\n{model_name}: dokładność={test_acc:.4f}, BA={bal_acc:.4f}, '
          f'F1m={f1_macro:.4f} (czas: {elapsed:.1f} min)')

    results = {
        'model': model_name,
        'fold': FOLD_IDX,
        'use_class_weights': USE_CLASS_WEIGHTS,
        'test_loss': float(test_loss),
        'test_accuracy': float(test_acc),
        'balanced_accuracy': float(bal_acc),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_weighted),
        'best_val_acc_phase1': float(max(h1.history['val_accuracy'])),
        'best_val_acc_phase2': float(max(h2.history['val_accuracy'])),
        'epochs_phase1': len(h1.history['val_accuracy']),
        'epochs_phase2': len(h2.history['val_accuracy']),
        'training_time_min': float(elapsed),
        'class_names': class_names,
        'per_class': {
            class_names[i]: {
                'precision': float(precision[i]),
                'recall': float(recall[i]),
                'f1': float(f1_per_class[i]),
                'support': int(support[i]),
            } for i in range(NUM_CLASSES)
        },
        'history_phase1': {k: [float(x) for x in v] for k, v in h1.history.items()},
        'history_phase2': {k: [float(x) for x in v] for k, v in h2.history.items()},
    }

    # Zapis na Drive
    with open(f'{FOLD_DIR}/{model_name.lower()}_metrics.json', 'w') as f:
        json.dump(results, f, indent=2)

    np.savez(f'{FOLD_DIR}/{model_name.lower()}_predictions.npz',
             y_true=y_true, y_pred=y_pred, y_proba=y_proba,
             confusion_matrix=cm, class_names=np.array(class_names))

    if not SAVE_WEIGHTS:
        if os.path.exists(ckpt_path):
            os.remove(ckpt_path)

    return model, results


In [ ]:
# === TRENOWANIE RESNET50 ===
if os.path.exists(done_marker):
    print('Podział już ukończony w całości, pomijam.')
    model_resnet = None
    resnet_results = None
elif resnet_done:
    print('ResNet50 już wytrenowany, wczytuję metryki.')
    with open(f'{FOLD_DIR}/resnet50_metrics.json') as f:
        resnet_results = json.load(f)
    model_resnet = None
else:
    model_resnet, resnet_results = build_and_train(
        'ResNet50', ResNet50, preprocess_resnet, RESNET_UNFREEZE
    )


In [ ]:
# Zwolnienie pamięci między modelami
if model_resnet is not None:
    del model_resnet
tf.keras.backend.clear_session()
gc.collect()
print('Pamięć zwolniona.')


In [ ]:
# === TRENOWANIE VGG16 ===
if os.path.exists(done_marker):
    print('Podział już ukończony w całości, pomijam.')
    model_vgg = None
    vgg_results = None
elif vgg_done:
    print('VGG16 już wytrenowany, wczytuję metryki.')
    with open(f'{FOLD_DIR}/vgg16_metrics.json') as f:
        vgg_results = json.load(f)
    model_vgg = None
else:
    model_vgg, vgg_results = build_and_train(
        'VGG16', VGG16, preprocess_vgg, VGG_UNFREEZE
    )


In [ ]:
# === ZAPIS PODSUMOWANIA PODZIAŁU ===
if not os.path.exists(done_marker) and resnet_results and vgg_results:
    summary = {
        'fold_idx': FOLD_IDX,
        'n_folds': N_FOLDS,
        'random_seed': RANDOM_SEED,
        'n_train': len(df_train),
        'n_val': len(df_val),
        'n_test': len(df_test),
        'resnet50': {
            'test_accuracy': resnet_results['test_accuracy'],
            'balanced_accuracy': resnet_results['balanced_accuracy'],
            'f1_macro': resnet_results['f1_macro'],
            'f1_weighted': resnet_results['f1_weighted'],
        },
        'vgg16': {
            'test_accuracy': vgg_results['test_accuracy'],
            'balanced_accuracy': vgg_results['balanced_accuracy'],
            'f1_macro': vgg_results['f1_macro'],
            'f1_weighted': vgg_results['f1_weighted'],
        },
    }
    with open(done_marker, 'w') as f:
        json.dump(summary, f, indent=2)

    print('=' * 60)
    print(f'PODZIAŁ {FOLD_IDX+1} ZAKOŃCZONY')
    print('=' * 60)
    print(f'ResNet50: dokładność={summary["resnet50"]["test_accuracy"]:.4f}, '
          f'BA={summary["resnet50"]["balanced_accuracy"]:.4f}, '
          f'F1m={summary["resnet50"]["f1_macro"]:.4f}')
    print(f'VGG16:    dokładność={summary["vgg16"]["test_accuracy"]:.4f}, '
          f'BA={summary["vgg16"]["balanced_accuracy"]:.4f}, '
          f'F1m={summary["vgg16"]["f1_macro"]:.4f}')
    print()
    print('Następny krok:')
    print(f'  1. Zmień FOLD_IDX na {(FOLD_IDX + 1) % N_FOLDS}')
    print('  2. Runtime, Run all')
else:
    print('Podział już ukończony lub trening został pominięty.')


In [ ]:
# === LISTA PLIKÓW WYJŚCIOWYCH NA DRIVE ===
import subprocess
print(f'Pliki w {FOLD_DIR}:')
result = subprocess.run(['ls', '-lh', FOLD_DIR], capture_output=True, text=True)
print(result.stdout)
